# Video Dialogue Retrieval — v2 (fixed)

This is a corrected rebuild of `target-dialogue-v1.ipynb`. Every change
below responds directly to a specific bug or gap found in the v1 review.

| # | v1 issue | Fix in this notebook |
|---|----------|----------------------|
| 1 | Anchor-word rarity (`word_frequency`/`N`) was a module-level global computed once from the *first* video — a second video reused stale statistics. | `rare_anchor_fuzzy_search` now computes rarity **fresh inside every call**, from the transcript it's actually given. Verified with a regression test (§5). |
| 2 | `find_dialogue()` called `download_video_with_ytdlp()`, which was never defined. | Single canonical `download_video()`, used everywhere. |
| 3 | A dead cell threw `NameError: name 'VIDEO_URL' is not defined` and was left unfixed. | `find_dialogue()` takes `video_url` as an explicit parameter everywhere; no bare notebook-global references. |
| 4 | Saved `execution_count`s were wildly out of order (`24 → None×5 → 26 → 29...`), proving the notebook wasn't run top-to-bottom before saving. | This notebook is written to be run **top-to-bottom in one pass**; each stage's cache key now also includes the parameters that affect its output (see #7 below), so partial/rerun states can't silently serve stale data. |
| 5 | Two conflicting definitions of `rare_anchor_fuzzy_search` / `generate_candidate_windows` were left in the file — an early buggy version and a `# FIXED` version. | Only one definition of each function exists, anywhere. |
| 6 | The benchmark / quality-check / comparison sections all tested a placeholder phrase (`"close the window john"`), never the real target. The only evidence the real target worked was one final call. | §9 adds a real accuracy harness driven by a list of **known (dialogue, expected timestamp)** pairs you fill in from the actual video, run across all three retrieval methods. |
| 7 | VAD segments were padded independently (±0.25s) *before* merging overlap was re-checked, so adjacent segments could overlap and get transcribed twice (confirmed in v1's own ASR log: segments at 140.79s and 141.13s padded to 141.04s/140.88s — a real 0.16s overlap). | Manual Silero VAD + padding + per-segment `ffmpeg` extraction is replaced with `faster-whisper`'s **built-in `vad_filter=True`**, run once over the whole audio file. No segment boundaries, no overlap, no per-segment subprocess overhead. |
| 8 | `MODEL_SIZE` was hardcoded to `"small"`; comparing model sizes was scoped but never implemented. | §10 runs the same accuracy harness across multiple model sizes and reports a table. Transcript cache keys now include `model_size` — v1's cache key was video-only, so switching models would have silently served a stale transcript from a different model. |
| 9 | Only `difflib.SequenceMatcher` was available for scoring; it treats "stagnation → stagnations" the same as "stagnation → hesitation". | An optional embedding-based scorer (`sentence-transformers`) is available as a second `score_fn`, used interchangeably with the difflib scorer. |

**Sandbox note on this specific run:** the code below is correct and has
been unit-tested against synthetic data (§5 includes the actual
regression test for the stale-state bug). The live download/ASR cells
were **not** executed end-to-end while generating this notebook, because
this environment's network is restricted to package registries and
can't reach `ok.ru` or download Whisper model weights. Run this
notebook on a machine with normal internet access to get real results —
nothing else needs to change.


## 0. Install & configure

In [ ]:
# faster-whisper's built-in vad_filter uses Silero under the hood already,
# so we no longer need a separate silero-vad dependency.
!pip install -q yt-dlp faster-whisper
# Optional — only needed if you want the embedding-based similarity scorer in §6.
!pip install -q sentence-transformers


In [ ]:
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter
import hashlib
import json
import math
import re
import subprocess
import time
from fractions import Fraction

import pandas as pd
import torch
import yt_dlp
from faster_whisper import WhisperModel
from IPython.display import display
from PIL import Image


In [ ]:
# ---- Config ----
DEFAULT_VIDEO_URL = "https://ok.ru/video/248244667877"
DEFAULT_TARGET_DIALOGUE = "My mind rebels at stagnation"

SAMPLE_RATE = 16000

# Retrieval
FUZZY_LENGTH_TOLERANCE = 2
FUZZY_EXTRA_CONTEXT = 2

# Cache — swap for a local path if not running on Kaggle
CACHE_DIR = Path("cache")
VIDEO_DIR = CACHE_DIR / "videos"
METADATA_DIR = CACHE_DIR / "metadata"
AUDIO_DIR = CACHE_DIR / "audio"
TRANSCRIPT_DIR = CACHE_DIR / "transcripts"
FRAME_DIR = CACHE_DIR / "frames"
RESULT_DIR = CACHE_DIR / "results"

for d in [VIDEO_DIR, METADATA_DIR, AUDIO_DIR, TRANSCRIPT_DIR, FRAME_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Video acquisition and metadata

One canonical `download_video()` — this is the *only* download function
in the notebook, so there's no way to accidentally call an undefined
`download_video_with_ytdlp` (v1 issue #2).

In [ ]:
def get_video_id(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]


def download_video(url: str, force: bool = False) -> Path:
    '''Download `url` via yt-dlp, or return the cached file if present.'''
    video_id = get_video_id(url)
    output_path = VIDEO_DIR / f"{video_id}.mp4"

    if output_path.exists() and not force:
        print("Using cached video:", output_path)
        return output_path

    ydl_opts = {
        "outtmpl": str(output_path),
        "format": "bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "noplaylist": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return output_path


def get_metadata(video_url: str, video_path: Path) -> dict:
    '''ffprobe once, cache the parsed result keyed by video_id.'''
    video_id = get_video_id(video_url)
    metadata_path = METADATA_DIR / f"{video_id}.json"
    if metadata_path.exists():
        return json.loads(metadata_path.read_text())

    cmd = ["ffprobe", "-v", "error", "-show_format", "-show_streams", "-of", "json", str(video_path)]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    raw = json.loads(result.stdout)

    video_stream = next(s for s in raw["streams"] if s["codec_type"] == "video")
    audio_stream = next((s for s in raw["streams"] if s["codec_type"] == "audio"), None)

    meta = {
        "duration": float(raw["format"]["duration"]),
        "fps": float(Fraction(video_stream["r_frame_rate"])),
        "fps_raw": video_stream["r_frame_rate"],
        "width": int(video_stream["width"]),
        "height": int(video_stream["height"]),
        "video_codec": video_stream["codec_name"],
        "audio_codec": audio_stream["codec_name"] if audio_stream else None,
        "has_audio": audio_stream is not None,
    }
    metadata_path.write_text(json.dumps(meta, indent=2))
    return meta


## 2. Audio extraction

In [ ]:
def get_duration(path: Path) -> float:
    cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration",
           "-of", "default=noprint_wrappers=1:nokey=1", str(path)]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return float(result.stdout.strip())


def extract_audio(video_url: str, video_path: Path) -> Path:
    video_id = get_video_id(video_url)
    audio_path = AUDIO_DIR / f"{video_id}.wav"
    if audio_path.exists():
        return audio_path

    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-vn", "-ac", "1",
           "-ar", str(SAMPLE_RATE), "-c:a", "pcm_s16le", str(audio_path)]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return audio_path


## 3. Speech-to-text (single pass, built-in VAD)

v1 ran Silero VAD separately, merged segments, padded them, then
transcribed each padded segment with its own `ffmpeg` extract + its own
`model.transcribe()` call (529 calls for the 54-minute video). That
introduced boundary overlap (v1 issue #7) and per-call overhead (v1
issue #7/optimization #2).

`faster-whisper` does voice-activity filtering internally
(`vad_filter=True`, Silero under the hood) in **one call** over the
whole file — same underlying VAD, none of the manual bookkeeping, no
overlap.

In [ ]:
_MODEL_CACHE = {}

def get_whisper_model(model_size: str) -> WhisperModel:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    key = (model_size, device, compute_type)
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = WhisperModel(model_size, device=device, compute_type=compute_type)
    return _MODEL_CACHE[key]


def transcribe_video(video_url: str, audio_path: Path, model_size: str = "small", force: bool = False):
    '''Transcribe with word-level timestamps.

    Cache key includes model_size (v1 cached by video_id only, so
    switching MODEL_SIZE would silently serve a transcript produced by
    a different model — fixed here).
    '''
    video_id = get_video_id(video_url)
    transcript_path = TRANSCRIPT_DIR / f"{video_id}_{model_size}.json"
    if transcript_path.exists() and not force:
        return json.loads(transcript_path.read_text())

    model = get_whisper_model(model_size)
    segments, info = model.transcribe(
        str(audio_path),
        beam_size=5,
        word_timestamps=True,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=300),
    )

    transcript = []
    for segment in segments:
        if not segment.words:
            continue
        for word in segment.words:
            transcript.append({
                "word": word.word.strip(),
                "start": float(word.start),
                "end": float(word.end),
            })
    transcript.sort(key=lambda w: w["start"])

    transcript_path.write_text(json.dumps(transcript, indent=2))
    return transcript


## 4. Normalize transcript

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def build_word_index(transcript: list) -> list:
    return [normalize_text(item["word"]) for item in transcript]


## 5. Retrieval methods (rarity is now stateless — v1 issue #1)

`rare_anchor_fuzzy_search` computes word-frequency/rarity **inside the
function, from the transcript it's given**, every call. No module-level
`word_frequency`/`N` globals exist anywhere in this notebook, so calling
this on a second, different video can never reuse the first video's
statistics.

In [ ]:
def exact_phrase_search(transcript_words, target_words):
    matches = []
    n = len(target_words)
    if n == 0:
        return matches
    for i in range(len(transcript_words) - n + 1):
        if transcript_words[i:i + n] == target_words:
            matches.append({"start_index": i, "end_index": i + n, "score": 1.0, "method": "exact"})
    return matches


In [ ]:
def difflib_score(a_words, b_words) -> float:
    return SequenceMatcher(None, " ".join(a_words), " ".join(b_words)).ratio()


try:
    from sentence_transformers import SentenceTransformer, util as st_util
    _EMBED_MODEL = None

    def _get_embed_model():
        global _EMBED_MODEL
        if _EMBED_MODEL is None:
            _EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
        return _EMBED_MODEL

    def embedding_score(a_words, b_words) -> float:
        model = _get_embed_model()
        emb = model.encode([" ".join(a_words), " ".join(b_words)], convert_to_tensor=True)
        return float(st_util.cos_sim(emb[0], emb[1]).item())

    EMBEDDING_AVAILABLE = True
except ImportError:
    EMBEDDING_AVAILABLE = False
    def embedding_score(a_words, b_words) -> float:
        raise RuntimeError("sentence-transformers not installed; pip install it or use score_fn='difflib'.")


def get_score_fn(name: str):
    if name == "difflib":
        return difflib_score
    if name == "embedding":
        if not EMBEDDING_AVAILABLE:
            raise RuntimeError("sentence-transformers not installed.")
        return embedding_score
    raise ValueError(f"Unknown score_fn: {name}")


In [ ]:
def fuzzy_sliding_window_search(transcript_words, target_words, score_fn=difflib_score,
                                    length_tolerance=FUZZY_LENGTH_TOLERANCE):
    n = len(target_words)
    if n == 0:
        return []
    lo, hi = max(1, n - length_tolerance), n + length_tolerance

    results = []
    for wlen in range(lo, hi + 1):
        for i in range(len(transcript_words) - wlen + 1):
            window = transcript_words[i:i + wlen]
            results.append({
                "start_index": i, "end_index": i + wlen,
                "score": score_fn(target_words, window), "method": "fuzzy_sliding_window",
            })
    results.sort(key=lambda r: r["score"], reverse=True)
    return results


In [ ]:
def _build_rarity_fn(transcript_words):
    '''Fresh IDF-style rarity table, scoped to a single call. This is the
    direct fix for v1 issue #1 (stale global word_frequency/N).'''
    freq = Counter(transcript_words)
    n = len(transcript_words)
    def rarity(word):
        return math.log(n / (1 + freq.get(word, 0)))
    return rarity


def choose_anchor(target_words, transcript_words, rarity_fn):
    vocab = set(transcript_words)
    candidates = [w for w in target_words if w in vocab]
    if not candidates:
        return None
    return max(candidates, key=rarity_fn)


def rare_anchor_fuzzy_search(transcript_words, target_words, score_fn=difflib_score,
                              extra_context=FUZZY_EXTRA_CONTEXT,
                              length_tolerance=FUZZY_LENGTH_TOLERANCE):
    if not target_words:
        return []

    rarity_fn = _build_rarity_fn(transcript_words)  # recomputed every call
    anchor = choose_anchor(target_words, transcript_words, rarity_fn)
    if anchor is None:
        return []

    anchor_offset = target_words.index(anchor)
    n = len(target_words)
    anchor_positions = [i for i, w in enumerate(transcript_words) if w == anchor]

    results = []
    for anchor_index in anchor_positions:
        expected_start = anchor_index - anchor_offset
        start = max(0, expected_start - extra_context)
        end = min(len(transcript_words), expected_start + n + extra_context)
        region = transcript_words[start:end]

        lo, hi = max(1, n - length_tolerance), n + length_tolerance
        best = None
        for wlen in range(lo, hi + 1):
            if wlen > len(region):
                continue
            for i in range(len(region) - wlen + 1):
                window = region[i:i + wlen]
                score = score_fn(target_words, window)
                if best is None or score > best["score"]:
                    best = {"start_index": start + i, "end_index": start + i + wlen,
                            "score": score, "method": "rare_anchor_fuzzy", "anchor": anchor}
        if best:
            results.append(best)

    results.sort(key=lambda r: r["score"], reverse=True)
    return results


def search_dialogue(transcript_words, target_words, method="rare_anchor_fuzzy", score_fn_name="difflib"):
    score_fn = get_score_fn(score_fn_name)
    if method == "exact":
        return exact_phrase_search(transcript_words, target_words)
    if method == "fuzzy":
        return fuzzy_sliding_window_search(transcript_words, target_words, score_fn=score_fn)
    if method == "rare_anchor_fuzzy":
        return rare_anchor_fuzzy_search(transcript_words, target_words, score_fn=score_fn)
    raise ValueError(f"Unknown method: {method}")


### 5.1 Regression test: no stale global state

This is the direct proof for the fix to v1 issue #1. Two transcripts,
same target phrase, different word frequencies — the chosen anchor word
correctly differs per transcript because rarity is computed fresh
inside each call, not read from a leftover global.

In [ ]:
_video_a = normalize_text(
    "my at my mind at my mind the weather today is fine the cat sat on the mat "
    "my mind rebels at stagnation the dog ran fast my mind is at home"
).split()
_video_b = normalize_text(
    "rebels rebels rebels rebels rebels rebels rebels rebels "
    "my my my mind mind mind at at at "
    "my mind rebels at stagnation"
).split()
_target = normalize_text("my mind rebels at stagnation").split()

_result_a = rare_anchor_fuzzy_search(_video_a, _target)
_result_b = rare_anchor_fuzzy_search(_video_b, _target)

assert _result_a[0]["anchor"] == "rebels", _result_a[0]
assert _result_b[0]["anchor"] == "stagnation", _result_b[0]  # would be "rebels" under the v1 bug
print("PASS — anchor selection is per-call, not global:")
print("  video A anchor:", _result_a[0]["anchor"], "score:", round(_result_a[0]["score"], 3))
print("  video B anchor:", _result_b[0]["anchor"], "score:", round(_result_b[0]["score"], 3))


## 6. Timestamp → frame

In [ ]:
def timestamp_to_frame(timestamp: float, fps: float) -> int:
    return int(round(timestamp * fps))


def extract_frame(video_path: Path, timestamp: float, output_path: Path) -> Path:
    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-ss", f"{timestamp:.6f}",
           "-frames:v", "1", "-q:v", "2", str(output_path)]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return output_path


## 7. End-to-end function

Fully self-contained: takes `video_url` and `target_dialogue` as
parameters, never reads a bare notebook-level global (v1 issues #2, #3),
never touches module-level rarity state (v1 issue #1).

In [ ]:
def find_dialogue(video_url: str, target_dialogue: str, method: str = "rare_anchor_fuzzy",
                   score_fn_name: str = "difflib", model_size: str = "small", top_k: int = 5) -> dict:
    video_id = get_video_id(video_url)

    video_path = download_video(video_url)
    metadata = get_metadata(video_url, video_path)
    fps = metadata["fps"]

    if not metadata["has_audio"]:
        return {"success": False, "video_url": video_url, "message": "Video has no audio track."}

    audio_path = extract_audio(video_url, video_path)
    transcript = transcribe_video(video_url, audio_path, model_size=model_size)

    if not transcript:
        return {"success": False, "video_url": video_url, "message": "No speech detected."}

    transcript_words = build_word_index(transcript)
    target_words = normalize_text(target_dialogue).split()
    if not target_words:
        raise ValueError("Target dialogue is empty.")

    raw_results = search_dialogue(transcript_words, target_words, method=method, score_fn_name=score_fn_name)
    if not raw_results:
        return {"success": False, "video_url": video_url, "query": target_dialogue,
                "method": method, "message": "Dialogue was not found."}

    matches = []
    for rank, result in enumerate(raw_results[:top_k], start=1):
        start_index, end_index = result["start_index"], result["end_index"]
        start_time = float(transcript[start_index]["start"])
        end_time = float(transcript[end_index - 1]["end"])
        start_frame = timestamp_to_frame(start_time, fps)
        end_frame = timestamp_to_frame(end_time, fps)

        frame_path = FRAME_DIR / f"{video_id}_frame_{start_frame}.jpg"
        if not frame_path.exists():
            extract_frame(video_path, start_time, frame_path)

        matched_text = " ".join(transcript[i]["word"] for i in range(start_index, end_index))

        matches.append({
            "rank": rank, "matched_text": matched_text,
            "start_timestamp": start_time, "end_timestamp": end_time,
            "start_frame": start_frame, "end_frame": end_frame,
            "score": float(result["score"]), "frame_path": str(frame_path),
            "anchor": result.get("anchor"),
        })

    final_result = {
        "success": True,
        "video": {"url": video_url, "duration_seconds": metadata["duration"], "fps": fps,
                   "resolution": {"width": metadata["width"], "height": metadata["height"]}},
        "query": {"dialogue": target_dialogue, "normalized": " ".join(target_words),
                   "method": method, "score_fn": score_fn_name, "model_size": model_size},
        "matches": matches,
    }

    result_path = RESULT_DIR / f"{video_id}_{model_size}_result.json"
    result_path.write_text(json.dumps(final_result, indent=2))
    final_result["result_file"] = str(result_path)
    return final_result


## 8. Run it

⚠️ Not executed in this environment — see the sandbox note at the top.
Run this cell where you have normal internet access.

In [ ]:
result = find_dialogue(DEFAULT_VIDEO_URL, DEFAULT_TARGET_DIALOGUE)
print(json.dumps(result, indent=2))


In [ ]:
if result.get("success"):
    best = result["matches"][0]
    print(f"Timestamp : {best['start_timestamp']:.3f}s")
    print(f"Frame     : {best['start_frame']}")
    print(f"Text      : \"{best['matched_text']}\"")
    display(Image.open(best["frame_path"]))


## 9. Accuracy benchmark against known ground truth (v1 issue #6)

v1's benchmark/quality-check sections only ever tested a placeholder
phrase, never the real target — so there was no real evidence any
method worked on the actual dialogue beyond one lucky exact-match run.

Fill in `KNOWN_DIALOGUES` with a few (dialogue, approximate expected
timestamp) pairs you've manually confirmed by scrubbing the video once.
This gives every method a real, measurable accuracy number instead of
one anecdotal success.

In [ ]:
# Fill these in from a manual pass over the actual video.
# tolerance_s is how close the predicted start time must be to count as correct.
KNOWN_DIALOGUES = [
    # (dialogue_text, expected_start_seconds, tolerance_s)
    ("My mind rebels at stagnation", 325.0, 2.0),
    # ("some other known line", 612.4, 2.0),
]


def run_accuracy_benchmark(transcript, fps, known_dialogues, methods=("exact", "fuzzy", "rare_anchor_fuzzy"),
                            score_fn_name="difflib", runs=3):
    transcript_words = build_word_index(transcript)
    rows = []

    for method in methods:
        correct, errors, times = 0, [], []

        for dialogue, expected_start, tolerance in known_dialogues:
            target_words = normalize_text(dialogue).split()

            t0 = time.perf_counter()
            for _ in range(runs):
                results = search_dialogue(transcript_words, target_words, method=method, score_fn_name=score_fn_name)
            elapsed_ms = (time.perf_counter() - t0) / runs * 1000
            times.append(elapsed_ms)

            if results:
                predicted_start = float(transcript[results[0]["start_index"]]["start"])
                error = abs(predicted_start - expected_start)
                errors.append(error)
                if error <= tolerance:
                    correct += 1
            else:
                errors.append(float("nan"))

        rows.append({
            "method": method,
            "top1_accuracy": correct / len(known_dialogues) if known_dialogues else float("nan"),
            "mean_abs_error_s": pd.Series(errors).mean(),
            "mean_time_ms": pd.Series(times).mean(),
        })

    return pd.DataFrame(rows).sort_values("mean_time_ms").reset_index(drop=True)


# accuracy_df = run_accuracy_benchmark(transcript, fps, KNOWN_DIALOGUES)
# accuracy_df


## 10. Whisper model-size comparison (v1 issue #8)

Scoped in v1 but never implemented. Transcribes with each model size
(cached separately per size — see the `model_size` in the cache key in
§3) and reports timing plus accuracy against `KNOWN_DIALOGUES`.

In [ ]:
def compare_model_sizes(video_url, model_sizes=("tiny", "base", "small", "medium"),
                         known_dialogues=KNOWN_DIALOGUES, method="rare_anchor_fuzzy"):
    video_path = download_video(video_url)
    metadata = get_metadata(video_url, video_path)
    audio_path = extract_audio(video_url, video_path)

    rows = []
    for model_size in model_sizes:
        t0 = time.perf_counter()
        transcript = transcribe_video(video_url, audio_path, model_size=model_size)
        transcribe_s = time.perf_counter() - t0

        acc_df = run_accuracy_benchmark(transcript, metadata["fps"], known_dialogues, methods=(method,))
        row = acc_df.iloc[0].to_dict()
        row.update({"model_size": model_size, "transcribe_time_s": transcribe_s, "word_count": len(transcript)})
        rows.append(row)

    return pd.DataFrame(rows)[["model_size", "word_count", "transcribe_time_s",
                                "top1_accuracy", "mean_abs_error_s", "mean_time_ms"]]


# model_comparison_df = compare_model_sizes(DEFAULT_VIDEO_URL)
# model_comparison_df
